In [ ]:
from itertools import product

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from trainutils.datasets.preset import load_dataset

In [ ]:
DATASET = 'cifar10'
NOISE_TYPES = ['uniform', 'flip', 'instance']
NOISE_RATES = [10, 20, 30, 40]

In [ ]:
def scores(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    return acc, f1, prec, rec

def baseline_predictions(fp, n_samples):
    filter_idxs = np.load(fp)
    prediction = np.ones(n_samples)
    prediction[filter_idxs] = 0
    return prediction

In [ ]:
fp = f'/data/datasets/{DATASET}'
train_set, _ = load_dataset(fp)
labels_clean = np.array(train_set.targets)

n_samples = len(train_set)
n_classes = len(train_set.classes)

In [ ]:
df = pd.DataFrame()

In [ ]:
for noise_type, noise_rate in product(NOISE_TYPES, NOISE_RATES):
    print(noise_type, noise_rate)
    results = []

    # load noise
    fp = f'/data/noise/seed_19/{DATASET}_train_{noise_type}_{noise_rate}_noise.npy'
    labels_noisy = np.load(fp)
    groundtruth = np.not_equal(labels_clean, labels_noisy)

    # bgss top 1
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_bgss_top1.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # bgss top 5 mv
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_bgss_top5_vote.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # bgss top 5 mean
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_bgss_top5_mean.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # bgss top 10 mv
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_bgss_top10_vote.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # bgss top 10 mean
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_bgss_top10_mean.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # baseline top 1
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_baseline_top1.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # baseline top 5 mv
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_baseline_top5_vote.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # baseline top 5 mean
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_baseline_top5_mean.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # baseline top 10 mv
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_baseline_top10_vote.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # baseline top 10 mean
    fp = f'/data/models/standard/seed_19/{DATASET}_{noise_type}_{noise_rate}/filter_baseline_top10_mean.npy'
    prediction = baseline_predictions(fp, n_samples)
    results.append(scores(groundtruth, prediction))

    # plateau 50-200 mv
    fp = f'/data/models/plateau/seed_19/{DATASET}_{noise_type}_{noise_rate}/majority_vote_results_50-200.npy'
    prediction = np.load(fp)
    results.append(scores(groundtruth, prediction))

    # plateau 50-200 mean
    fp = f'/data/models/plateau/seed_19/{DATASET}_{noise_type}_{noise_rate}/ensemble_results_50-200.npy'
    prediction = np.load(fp)
    results.append(scores(groundtruth, prediction))

    # plateau 0.085 50-200 mv
    fp = f'/data/models/plateau_0_085/seed_19/{DATASET}_{noise_type}_{noise_rate}/majority_vote_results_50-200.npy'
    prediction = np.load(fp)
    results.append(scores(groundtruth, prediction))

    # plateau 0.085 50-200 mean
    fp = f'/data/models/plateau_0_085/seed_19/{DATASET}_{noise_type}_{noise_rate}/ensemble_results_50-200.npy'
    prediction = np.load(fp)
    results.append(scores(groundtruth, prediction))

    # fill dataframe
    columns = pd.MultiIndex.from_product([[f'{noise_type} {noise_rate}'], ['acc', 'f1', 'prec', 'rec']])
    temp_df = pd.DataFrame(results, columns=columns)
    df = pd.concat([df, temp_df], axis=1)

In [ ]:
df.round(3)

In [ ]:
df.to_csv(f'results_{DATASET}.csv', sep=';')